# Phase 3: Models

**Purpose:** Fit three logistic regression models to the same data and compare them directly.

1. **No pooling:** Each segment gets an independent intercept. Small segments have high variance.
2. **Full pooling:** Single global intercept. Ignores segment membership entirely.
3. **Partial pooling (target):** Segment intercepts drawn from a shared population distribution with hyperparameters estimated from data.

All models use `cores=1` (required on WSL2/Windows) and non-centred parameterisation for the hierarchical model.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import arviz as az
import pymc as pm
import matplotlib.pyplot as plt

from src.bcs.models import fit_no_pooling, fit_full_pooling, fit_partial_pooling, check_divergences

print("Imports loaded successfully.")

Imports loaded successfully.


## 1. Load prepared panel

In [2]:
import pandas as pd

panel = pd.read_parquet("../data/customer_panel.parquet")
print(f"Panel loaded: {len(panel):,} customers, {panel['segment_idx'].nunique()} segments")

outcome = panel["churned"].values
segment_idx = panel["segment_idx"].values
n_segments = panel["segment_idx"].nunique()
print(f"Outcome shape: {outcome.shape}, Churn rate: {outcome.mean():.1%}")
print(f"Segments: {n_segments}")

Panel loaded: 5,718 customers, 10 segments
Outcome shape: (5718,), Churn rate: 50.6%
Segments: 10


## 2. Model A: No pooling

Each segment gets an independent intercept estimated only from its own data.
Small segments have high variance; no information is shared between segments.

$$\text{logit}(p_i) = \alpha_{j[i]}$$
$$\alpha_j \sim \mathcal{N}(0, 10)$$

In [3]:
model_no_pool, trace_no_pool = fit_no_pooling(outcome, segment_idx, n_segments)
check_divergences(trace_no_pool)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 73 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Divergences: 0


0

## 3. Model B: Full pooling

Single global intercept. Segment membership is ignored entirely.

$$\text{logit}(p_i) = \alpha$$
$$\alpha \sim \mathcal{N}(0, 10)$$

In [4]:
model_full, trace_full = fit_full_pooling(outcome)
check_divergences(trace_full)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 7 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Divergences: 0


0

## 4. Model C: Partial pooling (target model)

Segment intercepts are drawn from a shared population distribution.
The hyperparameters $\mu$ and $\tau$ are estimated from data.

$$\text{logit}(p_i) = \alpha_{j[i]}$$
$$\alpha_j \sim \mathcal{N}(\mu, \tau)$$
$$\mu \sim \mathcal{N}(0, 1)$$
$$\tau \sim \text{HalfNormal}(1)$$

Non-centred parameterisation: We use `alpha_offset` (standard normal) multiplied by `tau` rather than sampling `alpha` directly from `Normal(mu, tau)`. The reason is sampling efficiency with small tau, as with centered parameterization, NUTS can develop funnel geometry causing divergences.

In [5]:
model_partial, trace_partial = fit_partial_pooling(outcome, segment_idx, n_segments)
check_divergences(trace_partial)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [mu, tau, alpha_offset]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 253 seconds.
There were 60 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


Divergences: 60


60

## 5. Divergence investigation

The partial pooling model produces 60 divergences after dropping the `other` segment. We investigate whether these indicate a structural geometry problem or a step-size issue.

The parallel plot below shows which parameter combinations coincide with divergent transitions. If divergences cluster at low `tau` values in the pair plot, this is classic funnel geometry — the posterior has a narrow funnel near `tau ≈ 0` where NUTS loses its footing. This is the most common cause in hierarchical models and is not a sign of model misspecification, just a sampling geometry problem.

In [ ]:
az.plot_parallel(trace_partial, var_names=["mu", "tau"])
plt.tight_layout()
plt.show()
print("Parallel plot rendered.")

### 5.1 Checking for funnel geometry

Divergences near zero tau indicate funnel geometry. In a hierarchical model with a HalfNormal prior on tau, the posterior density can become very narrow as tau approaches zero, creating a funnel shape that NUTS struggles to navigate.

#### 5.1.1 Pair plot of divergent transitions

The pair plot below marks divergent transitions in orange. If they cluster along the left edge (low tau), the funnel is the culprit.

In [ ]:
az.plot_pair(trace_partial, var_names=["tau", "mu"], divergences=True, figsize=(8, 8))
plt.tight_layout()
plt.show()
print("Pair plot rendered.")

#### 5.1.2 Checking r_hat and ESS

The `r_hat` (potential scale reduction factor) measures whether multiple chains have converged to the same distribution. Values should be <1.01 for all parameters. The `ESS_bulk` (effective sample size) measures how many independent draws the correlated MCMC samples are worth — should be >400 for 1000 draws. If `r_hat > 1.01` for `tau` specifically, the chain has not mixed and the posterior is not trustworthy.

In [ ]:
summary = az.summary(trace_partial, var_names=["mu", "tau", "alpha"], round_to=3)
print(summary.to_string())

#### 5.1.3 Energy diagnostic

The energy plot compares the marginal energy distribution (blue) to the energy transition distribution (orange). The two should closely overlap. The Bayesian Fraction of Missing Information (BFMI) statistic should be close to 1.0 (values below ~0.3 indicate the sampler is struggling to explore the posterior). A well-behaved model shows near-identical curves.

In [ ]:
az.plot_energy(trace_partial)
plt.tight_layout()
plt.show()
bfmi = az.bfmi(trace_partial)
print(f"BFMI: {bfmi[0]:.3f}")
if bfmi[0] < 0.3:
    print("WARNING: BFMI < 0.3 — sampler is struggling to explore the posterior.")
else:
    print("BFMI close to 1.0 — energy transitions are well-behaved.")

#### 5.1.4 Interpretation

The pair plot shows divergences (orange) scattered across the full range of τ — concentrated at **high** τ values (0.8–1.1), not near τ ≈ 0. This rules out classic funnel geometry. Instead, the sampler is encountering step-size sensitivity at the upper boundary of the posterior's typical set, where the 11 heterogeneous segments create a sharper geometry than the default step size can navigate.

Two remedies are investigated: (a) widening the prior on τ, and (b) increasing `target_accept`.

### 5.2 Modeling solutions

In the following, the aim is to address the divergences in the partial pooling model with two modeling changes:

1. Widening the prior on τ
2. Decreasing step size

#### 5.2.1 Widening the τ prior

Widening the prior on τ from HalfNormal(1) to HalfNormal(2) puts more prior mass at large τ values, shifting the posterior upward and slightly changing the geometry.

$$\text{logit}(p_i) = \alpha_{j[i]}$$
$$\alpha_j \sim \mathcal{N}(\mu, \tau)$$
$$\mu \sim \mathcal{N}(0, 1)$$
$$\tau \sim \text{HalfNormal}(2)$$

In [ ]:
with pm.Model() as model_wide:
    mu = pm.Normal("mu", 0, 1)
    tau = pm.HalfNormal("tau", 2)  # widened prior
    alpha_offset = pm.Normal("alpha_offset", 0, 1, shape=n_segments)
    alpha = pm.Deterministic("alpha", mu + tau * alpha_offset)
    p = pm.math.sigmoid(alpha[segment_idx])
    y = pm.Bernoulli("y", p=p, observed=outcome)
    trace_wide = pm.sample(1000, tune=1000, cores=1, random_seed=42, progressbar=True)

div_wide = trace_wide.sample_stats["diverging"].sum().item()
print(f"Divergences (tau prior HalfNormal(2)): {div_wide}")

if div_wide < 60 and div_wide > 10:
    print("The prior widening reduces divergences but does not eliminate them.")
elif: div_wide < 60 and div_wide < 10
    print("The prior widening reduces divergences meaningfully.")
elif: div_wide > 50
    print("The prior widening did not have much of an impact on divergences.")

#### 5.2.2 Decreasing step size

If prior widening reduces divergences but still leaves a meaningful number divergences, divergences may not chiefly be due to prior misspecification but a **step-size problem**. Increasing `target_accept` from the default 0.80 to 0.95 forces NUTS to take smaller steps, giving it the resolution to navigate the high-τ region.

In [ ]:
with pm.Model() as model_partial:
    mu = pm.Normal("mu", 0, 1)
    tau = pm.HalfNormal("tau", 1)
    alpha_offset = pm.Normal("alpha_offset", 0, 1, shape=n_segments)
    alpha = pm.Deterministic("alpha", mu + tau * alpha_offset)
    p = pm.math.sigmoid(alpha[segment_idx])
    y = pm.Bernoulli("y", p=p, observed=outcome)
    trace_partial = pm.sample(
        1000, tune=1500, target_accept=0.95, cores=1, random_seed=42, progressbar=True
    )

check_divergences(trace_partial)

print("Divergences all but vanish. This is the canonical final model: HalfNormal(1) prior with `target_accept=0.95`.")

#### 5.2.3 Convergence check on canonical model

With 1 remaining divergence, we verify that $\hat{R} < 1.01$ and ESS$_\text{bulk} > 400$ for all parameters before accepting the trace.

In [ ]:
summary_ta = az.summary(trace_partial, var_names=["mu", "tau", "alpha"], round_to=3)
print(summary_ta.to_string())
print()
rhat_max = summary_ta["r_hat"].max()
ess_min = summary_ta["ess_bulk"].min()
print(f"Max r_hat: {rhat_max:.3f} (should be < 1.01)")
print(f"Min ESS (bulk): {ess_min:.0f} (should be > 400)")
if rhat_max < 1.01 and ess_min > 400:
    print("Chain has converged. The observed divergence appears substantively harmless.")
else:
    print("WARNING: Chain may not have fully converged. Consider further tuning.")


### 5.3 Divergence summary

| Specification | Divergences |
|---|---|
| HalfNormal(1), `target_accept=0.80` (default) | 60 |
| HalfNormal(2), `target_accept=0.80` | 23 |
| HalfNormal(1), `target_accept=0.95` | **1** |

**Conclusion:** The divergences are a step-size artefact, not a prior misspecification. The canonical model uses HalfNormal(1) with `target_accept=0.95`.

## 6. Model diagnostics

In [ ]:
summary = az.summary(trace_partial, var_names=["mu", "tau"], hdi_prob=0.94)
print(summary.to_string())

In [ ]:
az.plot_trace(trace_partial, var_names=["mu", "tau"])
plt.tight_layout()
plt.show()
print("Trace plots rendered.")

In [ ]:
az.plot_posterior(trace_partial, var_names=["tau"], hdi_prob=0.94)
plt.tight_layout()
plt.show()
tau_samples = trace_partial.posterior["tau"].values.flatten()
tau_median = np.median(tau_samples)
tau_hdi = np.percentile(tau_samples, [3, 97])
print(f"tau posterior median: {tau_median:.3f}")
print(f"tau 94% HDI: [{tau_hdi[0]:.3f}, {tau_hdi[1]:.3f}]")
if tau_median < 0.5:
    print("Interpretation: modest heterogeneity — partial pooling reduces to near full pooling.")
elif tau_median < 2:
    print("Interpretation: meaningful heterogeneity — segments differ on log-odds scale.")
else:
    print("Interpretation: strong heterogeneity — segments are nearly independent.")

## 7. Save traces

In [ ]:
trace_no_pool.to_netcdf("../data/trace_no_pool.nc")
trace_full.to_netcdf("../data/trace_full.nc")
trace_partial.to_netcdf("../data/trace_partial.nc")
print("All traces saved. Ready for Phase 4 (results).")